**Evaluate the three frozen models on the official 10-well BLIND set, compare their metrics, and visualize predictions depth-by-depth against the true lithology.**

# Final Blind-Test Evaluation — FORCE 2020 Lithology Prediction

## Objective

This notebook presents the final evaluation of the three frozen lithology
classification models on the official FORCE 2020 blind-test wells.

### Models evaluated

1. W5 — ExtraTrees with sqrt-balanced class weights
2. XGBoost — tuned gradient-boosted trees
3. LightGBM — tuned gradient-boosted trees

### Evaluation protocol

The official FORCE 2020 split is used:

- TRAIN: 98 wells
- PUBLIC: 10 wells
- BLIND: 10 wells

The BLIND wells were not used for model training or model selection.

The evaluation uses the same 41-feature representation used to train the
final models.

### Main objectives

- Compare the three models using the same blind samples.
- Evaluate overall and per-class performance.
- Calculate the official FORCE 2020 penalty score.
- Examine predictions well-by-well.
- Visualize true and predicted lithology as a function of depth.
- Identify the geological/lithological boundaries where models disagree.
- Determine which model provides the strongest overall blind-test performance.

> The blind set is treated as a locked evaluation set. No model or
> hyperparameter decisions are made using these results.

In [1]:
from pathlib import Path
import json
import joblib
import numpy as np
import pandas as pd

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
)

PROJECT_ROOT = Path.cwd().parent

DATA_DIR = PROJECT_ROOT / "data"
MODEL_DIR = PROJECT_ROOT / "models"

print("Project root:", PROJECT_ROOT)
print("Data directory:", DATA_DIR)
print("Model directory:", MODEL_DIR)

Project root: /Users/akashray/Lithology_Predictor
Data directory: /Users/akashray/Lithology_Predictor/data
Model directory: /Users/akashray/Lithology_Predictor/models


# importing the split ...that is made during the splitting the wells in training and testing

In [2]:
split_path = DATA_DIR / "splits" / "force2020_split.csv"

split_df = pd.read_csv(split_path)

print("Split shape:", split_df.shape)
print("\nSplit counts:")
print(split_df["SPLIT"].value_counts())

print("\nBLIND wells:")
blind_wells = split_df.loc[
    split_df["SPLIT"].str.upper() == "BLIND", "WELL"
].tolist()

for well in blind_wells:
    print(" ", well)

print("\nNumber of BLIND wells:", len(blind_wells))

Split shape: (118, 2)

Split counts:
SPLIT
TRAIN         98
PUBLIC        10
BLIND_TEST    10
Name: count, dtype: int64

BLIND wells:

Number of BLIND wells: 0


**loadinfg the features that we had made to train the model**

In [4]:
features_path = DATA_DIR / "processed" / "engineered_features.parquet"

df = pd.read_parquet(features_path)

print("Dataset shape:", df.shape)
print("\nColumns:", len(df.columns))
print("\nFirst columns:")
# print(df.columns.tolist())
for col in df.columns[:]:
    print(" ", col)

Dataset shape: (1430974, 46)

Columns: 46

First columns:
  GR
  RDEP
  RMED
  DTC
  RHOB
  FORCE_2020_LITHOFACIES_LITHOLOGY
  WELL
  RDEP_LOG10
  RMED_LOG10
  GR_MISSING
  RDEP_LOG10_MISSING
  RMED_LOG10_MISSING
  DTC_MISSING
  RHOB_MISSING
  SPLIT
  GR_ROLLMEAN_5
  GR_ROLLSTD_5
  GR_ROLLMEAN_9
  GR_ROLLSTD_9
  GR_ROLLMEAN_21
  GR_ROLLSTD_21
  RDEP_LOG10_ROLLMEAN_5
  RDEP_LOG10_ROLLSTD_5
  RDEP_LOG10_ROLLMEAN_9
  RDEP_LOG10_ROLLSTD_9
  RDEP_LOG10_ROLLMEAN_21
  RDEP_LOG10_ROLLSTD_21
  RMED_LOG10_ROLLMEAN_5
  RMED_LOG10_ROLLSTD_5
  RMED_LOG10_ROLLMEAN_9
  RMED_LOG10_ROLLSTD_9
  RMED_LOG10_ROLLMEAN_21
  RMED_LOG10_ROLLSTD_21
  DTC_ROLLMEAN_5
  DTC_ROLLSTD_5
  DTC_ROLLMEAN_9
  DTC_ROLLSTD_9
  DTC_ROLLMEAN_21
  DTC_ROLLSTD_21
  RHOB_ROLLMEAN_5
  RHOB_ROLLSTD_5
  RHOB_ROLLMEAN_9
  RHOB_ROLLSTD_9
  RHOB_ROLLMEAN_21
  RHOB_ROLLSTD_21
  RESISTIVITY_SEPARATION


**Load the three model metadata files**

In [16]:


model_paths = {
    "W5": MODEL_DIR / "lithology_w5_final.joblib",
    "XGBoost": MODEL_DIR / "lithology_xgboost_final.joblib",
    "LightGBM": MODEL_DIR / "lithology_lightgbm_final.joblib",
}

models = {}

for name, path in model_paths.items():
    print(f"Loading {name}...")
    models[name] = joblib.load(path)

    print("  Type:", type(models[name]).__name__)
    print("  Number of features:", models[name].n_features_in_)
    print("  Classes:", models[name].classes_)
    print()

print("All three models loaded successfully.")

Loading W5...
  Type: ExtraTreesClassifier
  Number of features: 41
  Classes: [30000. 65000. 65030. 70000. 70032. 74000. 80000. 86000. 88000. 90000.
 93000. 99000.]

Loading XGBoost...
  Type: XGBClassifier
  Number of features: 41
  Classes: [ 0  1  2  3  4  5  6  7  8  9 10 11]

Loading LightGBM...
  Type: LGBMClassifier
  Number of features: 41
  Classes: [ 0  1  2  3  4  5  6  7  8  9 10 11]

All three models loaded successfully.


**just verifyin the model are the right one and had trained on the desired engg. features...**

In [7]:
# ---------------------------------------------------------
# Step 6 — Verify the final 41-feature schema
# ---------------------------------------------------------

feature_lists = {
    name: meta["features"]
    for name, meta in metadata.items()
}

# Compare feature lists across all three models
w5_features = feature_lists["W5"]
xgb_features = feature_lists["XGBoost"]
lgbm_features = feature_lists["LightGBM"]

print("W5 features:", len(w5_features))
print("XGBoost features:", len(xgb_features))
print("LightGBM features:", len(lgbm_features))

print("\nFeature schemas identical?")
print("W5 == XGBoost :", w5_features == xgb_features)
print("W5 == LightGBM:", w5_features == lgbm_features)
print("XGBoost == LightGBM:", xgb_features == lgbm_features)

# Check that every required feature actually exists
missing_features = [
    f for f in w5_features
    if f not in df.columns
]

print("\nMissing required features:", missing_features)

# Display the authoritative 41-feature order
print("\nFinal 41-feature order:")
for i, feature in enumerate(w5_features, start=1):
    print(f"{i:2d}. {feature}")

W5 features: 41
XGBoost features: 41
LightGBM features: 41

Feature schemas identical?
W5 == XGBoost : True
W5 == LightGBM: True
XGBoost == LightGBM: True

Missing required features: []

Final 41-feature order:
 1. GR
 2. RDEP_LOG10
 3. RMED_LOG10
 4. DTC
 5. RHOB
 6. GR_ROLLMEAN_5
 7. GR_ROLLSTD_5
 8. GR_ROLLMEAN_9
 9. GR_ROLLSTD_9
10. GR_ROLLMEAN_21
11. GR_ROLLSTD_21
12. RDEP_LOG10_ROLLMEAN_5
13. RDEP_LOG10_ROLLSTD_5
14. RDEP_LOG10_ROLLMEAN_9
15. RDEP_LOG10_ROLLSTD_9
16. RDEP_LOG10_ROLLMEAN_21
17. RDEP_LOG10_ROLLSTD_21
18. RMED_LOG10_ROLLMEAN_5
19. RMED_LOG10_ROLLSTD_5
20. RMED_LOG10_ROLLMEAN_9
21. RMED_LOG10_ROLLSTD_9
22. RMED_LOG10_ROLLMEAN_21
23. RMED_LOG10_ROLLSTD_21
24. DTC_ROLLMEAN_5
25. DTC_ROLLSTD_5
26. DTC_ROLLMEAN_9
27. DTC_ROLLSTD_9
28. DTC_ROLLMEAN_21
29. DTC_ROLLSTD_21
30. RHOB_ROLLMEAN_5
31. RHOB_ROLLSTD_5
32. RHOB_ROLLMEAN_9
33. RHOB_ROLLSTD_9
34. RHOB_ROLLMEAN_21
35. RHOB_ROLLSTD_21
36. RESISTIVITY_SEPARATION
37. GR_MISSING
38. RDEP_LOG10_MISSING
39. RMED_LOG10_MISSIN

**Extract and verify the  BLIND data**

In [12]:
# Official FORCE 2020 blind-test wells
blind_wells = split_df.loc[
    split_df["SPLIT"].str.upper() == "BLIND_TEST",
    "WELL"
].tolist()

print("Number of BLIND_TEST wells:", len(blind_wells))
print("\nBLIND_TEST wells:")

for well in blind_wells:
    print(" ", well)

Number of BLIND_TEST wells: 10

BLIND_TEST wells:
  15_9-23
  16_2-7
  16_7-6
  17_4-1
  25_10-9
  31_2-10
  31_2-21 S
  34_3-2 S
  35_11-5
  35_9-7


In [17]:
# ---------------------------------------------------------
# Step 9 — Final blind-input verification
# ---------------------------------------------------------

# Load the label mapping used by XGBoost and LightGBM
mapping_path = MODEL_DIR / "boosted_label_mapping.json"

with open(mapping_path, "r") as f:
    boosted_mapping = json.load(f)

print("Boosted-model label mapping:")
print(boosted_mapping)

# Build the actual ML matrix in the exact frozen feature order
X_blind = blind_df[w5_features].copy()
y_blind = blind_df[target_col].copy()

print("\nX_blind shape:", X_blind.shape)
print("y_blind shape:", y_blind.shape)

# Check feature order and missing values
print("\nNaN values in X_blind:", int(X_blind.isna().sum().sum()))
print("Infinite values in X_blind:",
      int(np.isinf(X_blind.to_numpy(dtype=float)).sum()))

print("\nTrue lithology classes in BLIND:")
print(sorted(y_blind.unique()))

Boosted-model label mapping:
{'code_to_idx': {'30000': 0, '65000': 1, '65030': 2, '70000': 3, '70032': 4, '74000': 5, '80000': 6, '86000': 7, '88000': 8, '90000': 9, '93000': 10, '99000': 11}, 'idx_to_code': {'0': 30000, '1': 65000, '2': 65030, '3': 70000, '4': 70032, '5': 74000, '6': 80000, '7': 86000, '8': 88000, '9': 90000, '10': 93000, '11': 99000}}

X_blind shape: (122576, 41)
y_blind shape: (122576,)

NaN values in X_blind: 175190
Infinite values in X_blind: 0

True lithology classes in BLIND:
[np.float64(30000.0), np.float64(65000.0), np.float64(65030.0), np.float64(70000.0), np.float64(70032.0), np.float64(74000.0), np.float64(80000.0), np.float64(86000.0), np.float64(88000.0), np.float64(90000.0), np.float64(99000.0)]


In [25]:
# ---------------------------------------------------------
# Step 10 — Confirm all frozen models accept the
#           same NaN-containing 41-feature input
# ---------------------------------------------------------

X_probe = X_blind.iloc[:100]

print("Probe shape:", X_probe.shape)
print("Probe NaN count:", X_probe.isna().sum().sum())

for name, model in models.items():
    print(f"\nTesting {name}...")

    try:
        pred = model.predict(X_probe)

        print("  ✓ SUCCESS")
        print("  Prediction shape:", pred.shape)
        print("  First 10 predictions:", pred[:10])

    except Exception as e:
        print("  ✗ FAILED")
        print("  Error:", type(e).__name__)
        print("  Message:", str(e))

Probe shape: (100, 41)
Probe NaN count: 378

Testing W5...
  ✓ SUCCESS
  Prediction shape: (100,)
  First 10 predictions: [65000. 65000. 65000. 65000. 65000. 65000. 65000. 65000. 65000. 65000.]

Testing XGBoost...
  ✓ SUCCESS
  Prediction shape: (100,)
  First 10 predictions: [1 1 1 3 1 1 1 1 1 1]

Testing LightGBM...
  ✓ SUCCESS
  Prediction shape: (100,)
  First 10 predictions: [1 1 1 1 1 1 1 1 1 1]


In [26]:
# ---------------------------------------------------------
# Step 11 — Full BLIND prediction
# ---------------------------------------------------------

print("Running predictions on official BLIND_TEST set...")
print("Samples:", len(X_blind))
print("Features:", X_blind.shape[1])

blind_predictions = {}

# -------------------------
# W5 ExtraTrees
# -------------------------
print("\n[1/3] W5 ExtraTrees...")
w5_pred = models["W5"].predict(X_blind)

blind_predictions["W5"] = w5_pred.astype(int)

print("Done.")
print("Unique predicted classes:", np.unique(blind_predictions["W5"]))


# -------------------------
# XGBoost
# -------------------------
print("\n[2/3] XGBoost...")
xgb_pred_idx = models["XGBoost"].predict(X_blind)

xgb_pred = np.array([
    boosted_mapping["idx_to_code"][str(int(idx))]
    for idx in xgb_pred_idx
])

blind_predictions["XGBoost"] = xgb_pred.astype(int)

print("Done.")
print("Unique predicted classes:", np.unique(blind_predictions["XGBoost"]))


# -------------------------
# LightGBM
# -------------------------
print("\n[3/3] LightGBM...")
lgbm_pred_idx = models["LightGBM"].predict(X_blind)

lgbm_pred = np.array([
    boosted_mapping["idx_to_code"][str(int(idx))]
    for idx in lgbm_pred_idx
])

blind_predictions["LightGBM"] = lgbm_pred.astype(int)

print("Done.")
print("Unique predicted classes:", np.unique(blind_predictions["LightGBM"]))


print("\n" + "=" * 60)
print("ALL BLIND PREDICTIONS COMPLETED")
print("=" * 60)

for name, pred in blind_predictions.items():
    print(f"{name:10s} → {len(pred):,} predictions")

Running predictions on official BLIND_TEST set...
Samples: 122576
Features: 41

[1/3] W5 ExtraTrees...
Done.
Unique predicted classes: [30000 65000 65030 70000 70032 74000 80000 86000 88000 90000 93000 99000]

[2/3] XGBoost...
Done.
Unique predicted classes: [30000 65000 65030 70000 70032 74000 80000 86000 88000 90000 99000]

[3/3] LightGBM...
Done.
Unique predicted classes: [30000 65000 65030 70000 70032 74000 80000 86000 88000 90000 93000 99000]

ALL BLIND PREDICTIONS COMPLETED
W5         → 122,576 predictions
XGBoost    → 122,576 predictions
LightGBM   → 122,576 predictions


In [32]:
# ---------------------------------------------------------
# Step 12 — Build unified BLIND prediction table
# ---------------------------------------------------------

blind_results = blind_df[["WELL", target_col]].copy()

blind_results = blind_results.rename(
    columns={
        target_col: "TRUE_LITHOLOGY"
    }
)

blind_results["W5_PREDICTION"] = blind_predictions["W5"]
blind_results["XGBOOST_PREDICTION"] = blind_predictions["XGBoost"]
blind_results["LIGHTGBM_PREDICTION"] = blind_predictions["LightGBM"]

print("Blind results shape:", blind_results.shape)

display(blind_results)

Blind results shape: (122576, 5)


,WELL,TRUE_LITHOLOGY,W5_PREDICTION,XGBOOST_PREDICTION,LIGHTGBM_PREDICTION
73674,15_9-23,65000.0,65000,65000,65000
73675,15_9-23,65000.0,65000,65000,65000
73676,15_9-23,65000.0,65000,65000,65000
73677,15_9-23,65000.0,65000,70000,65000
73678,15_9-23,65000.0,65000,65000,65000
...,...,...,...,...,...
1404784,35_9-7,65000.0,65000,65030,65000
1404785,35_9-7,65000.0,65000,65000,65000
1404786,35_9-7,65000.0,65000,65000,65000
1404787,35_9-7,65000.0,65000,65000,65000


In [31]:
print("\nMissing values:")
print(blind_results.isna().sum())

print("\nNumber of mismatches with TRUE lithology:")

for model_name, column in [
    ("W5", "W5_PREDICTION"),
    ("XGBoost", "XGBOOST_PREDICTION"),
    ("LightGBM", "LIGHTGBM_PREDICTION"),
]:
    errors = (blind_results["TRUE_LITHOLOGY"] != blind_results[column]).sum()
    print(f"{model_name:10s}: {errors:,}")


Missing values:
WELL                   0
TRUE_LITHOLOGY         0
W5_PREDICTION          0
XGBOOST_PREDICTION     0
LIGHTGBM_PREDICTION    0
dtype: int64

Number of mismatches with TRUE lithology:
W5        : 32,032
XGBoost   : 39,628
LightGBM  : 36,129


In [34]:
# Find model-related variables currently available in the notebook

[name for name in globals()
 if any(word in name.lower() for word in ["xgb", "model", "boosted"])]

['MODEL_DIR',
 'xgb_features',
 'model_paths',
 'models',
 'boosted_mapping',
 'model',
 'xgb_pred_idx',
 'xgb_pred',
 'model_name']

In [35]:
print("XGB prediction index distribution:")
print(pd.Series(xgb_pred_idx).value_counts().sort_index())

print("\nXGB decoded prediction distribution:")
print(pd.Series(xgb_pred).value_counts().sort_index())

print("\nXGB model feature count:")
print(len(xgb_features))

print("\nMapping:")
print(boosted_mapping)

XGB prediction index distribution:
0     15711
1     71491
2     10394
3      9988
4      1296
5        39
6      5918
7      4579
8      1657
9       287
11     1216
Name: count, dtype: int64

XGB decoded prediction distribution:
30000    15711
65000    71491
65030    10394
70000     9988
70032     1296
74000       39
80000     5918
86000     4579
88000     1657
90000      287
99000     1216
Name: count, dtype: int64

XGB model feature count:
41

Mapping:
{'code_to_idx': {'30000': 0, '65000': 1, '65030': 2, '70000': 3, '70032': 4, '74000': 5, '80000': 6, '86000': 7, '88000': 8, '90000': 9, '93000': 10, '99000': 11}, 'idx_to_code': {'0': 30000, '1': 65000, '2': 65030, '3': 70000, '4': 70032, '5': 74000, '6': 80000, '7': 86000, '8': 88000, '9': 90000, '10': 93000, '11': 99000}}


In [36]:
print("models object:")
print(type(models))

if isinstance(models, dict):
    print("\nModel keys:")
    print(models.keys())
    print("\nXGBoost object:")
    print(type(models.get("XGBoost")))
    print(models.get("XGBoost"))

print("\nmodel_paths:")
print(model_paths)

print("\nxgb_features:")
print(len(xgb_features))
print(xgb_features)

models object:
<class 'dict'>

Model keys:
dict_keys(['W5', 'XGBoost', 'LightGBM'])

XGBoost object:
<class 'xgboost.sklearn.XGBClassifier'>
XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=0.8, device=None, early_stopping_rounds=None,
              enable_categorical=True, eval_metric='mlogloss',
              feature_types=None, feature_weights=None, gamma=None,
              grow_policy=None, importance_type=None,
              interaction_constraints=None, learning_rate=0.05, max_bin=None,
              max_cat_threshold=None, max_cat_to_onehot=None,
              max_delta_step=None, max_depth=8, max_leaves=None,
              min_child_weight=None, missing=nan, monotone_constraints=None,
              multi_strategy=None, n_estimators=500, n_jobs=-1, num_class=12, ...)

model_paths:
{'W5': PosixPath('/Users/akashray/Lithology_Predictor/models/lithology_w5_final.joblib'), 'XGBoo

In [37]:
# Directly predict with the frozen XGBoost model
xgb_model = models["XGBoost"]

direct_xgb_idx = xgb_model.predict(X_blind).astype(int)

print("Existing xgb_pred_idx == direct prediction:",
      np.array_equal(xgb_pred_idx, direct_xgb_idx))

print("\nDirect XGBoost accuracy (class indices):")
print(np.mean(direct_xgb_idx == np.array([
    boosted_mapping["code_to_idx"][str(int(v))]
    for v in y_blind
])))

print("\nExisting XGBoost accuracy (decoded codes):")
print(np.mean(xgb_pred == y_blind))

print("\nNumber of different predictions:",
      np.sum(xgb_pred_idx != direct_xgb_idx))

Existing xgb_pred_idx == direct prediction: True

Direct XGBoost accuracy (class indices):
0.6767066962537528

Existing XGBoost accuracy (decoded codes):
0.6767066962537528

Number of different predictions: 0


In [39]:
# Search stored notebook variables safely

for name, value in list(globals().items()):
    if "xgb" in name.lower() or "metric" in name.lower() or "result" in name.lower():
        if isinstance(value, (float, int, np.floating, np.integer, dict, pd.DataFrame, pd.Series)):
            print("\n---", name, "---")
            print(value)


--- blind_results ---
            WELL  TRUE_LITHOLOGY  W5_PREDICTION  XGBOOST_PREDICTION  \
73674    15_9-23         65000.0          65000               65000   
73675    15_9-23         65000.0          65000               65000   
73676    15_9-23         65000.0          65000               65000   
73677    15_9-23         65000.0          65000               70000   
73678    15_9-23         65000.0          65000               65000   
...          ...             ...            ...                 ...   
1404784   35_9-7         65000.0          65000               65030   
1404785   35_9-7         65000.0          65000               65000   
1404786   35_9-7         65000.0          65000               65000   
1404787   35_9-7         65000.0          65000               65000   
1404788   35_9-7         65000.0          65000               65000   

         LIGHTGBM_PREDICTION  
73674                  65000  
73675                  65000  
73676                  65000  


In [40]:
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    classification_report,
)

evaluation_rows = []

for model_name, pred in blind_predictions.items():
    evaluation_rows.append({
        "Model": model_name,
        "Accuracy": accuracy_score(y_blind, pred),
        "Balanced Accuracy": balanced_accuracy_score(y_blind, pred),
        "Macro F1": f1_score(
            y_blind, pred,
            average="macro",
            zero_division=0
        ),
        "Weighted F1": f1_score(
            y_blind, pred,
            average="weighted",
            zero_division=0
        ),
    })

metrics_df = pd.DataFrame(evaluation_rows)

print(metrics_df.to_string(index=False))

/Users/akashray/Lithology_Predictor/.venv/lib/python3.13/site-packages/sklearn/metrics/_classification.py:2939: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")


   Model  Accuracy  Balanced Accuracy  Macro F1  Weighted F1
      W5  0.738676           0.463725  0.433924     0.713718
 XGBoost  0.676707           0.431236  0.391873     0.680074
LightGBM  0.705252           0.482206  0.451275     0.703439


/Users/akashray/Lithology_Predictor/.venv/lib/python3.13/site-packages/sklearn/metrics/_classification.py:2939: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")


In [41]:
# Load depth only for the official blind wells

v2_path = DATA_DIR / "processed" / "engineered_features_v2.parquet"

v2 = pd.read_parquet(v2_path)

print("V2 shape:", v2.shape)
print("Has DEPT:", "DEPT" in v2.columns)

blind_depth = v2.loc[
    v2["WELL"].isin(blind_wells),
    ["WELL", "DEPT"]
].copy()

print("\nBlind depth shape:", blind_depth.shape)
print("Blind wells:", blind_depth["WELL"].nunique())
print("\nSamples per well:")
print(blind_depth["WELL"].value_counts().sort_index())

V2 shape: (1430974, 52)
Has DEPT: True

Blind depth shape: (122576, 2)
Blind wells: 10

Samples per well:
WELL
15_9-23      11063
16_2-7       11683
16_7-6       10222
17_4-1       17271
25_10-9      10788
31_2-10       9033
31_2-21 S     7840
34_3-2 S     12229
35_11-5      21770
35_9-7       10677
Name: count, dtype: int64


In [42]:
# Verify that blind_depth is in the exact same WELL order as blind_results

same_well_order = (
    blind_depth["WELL"].reset_index(drop=True)
    .equals(blind_results["WELL"].reset_index(drop=True))
)

print("Same WELL row order:", same_well_order)

# Check total number of rows
print("blind_results rows:", len(blind_results))
print("blind_depth rows:", len(blind_depth))

# If the order matches, safely attach depth
if same_well_order:
    blind_results = blind_results.copy()
    blind_results["DEPT"] = blind_depth["DEPT"].reset_index(drop=True)

    print("\nDEPT successfully attached.")
    print(blind_results.head())
    print("\nDepth range:")
    print(blind_results.groupby("WELL")["DEPT"].agg(["min", "max", "count"]))
else:
    print("\nWARNING: WELL row order does not match.")
    print("Do NOT attach DEPT yet.")

Same WELL row order: True
blind_results rows: 122576
blind_depth rows: 122576

DEPT successfully attached.
          WELL  TRUE_LITHOLOGY  W5_PREDICTION  XGBOOST_PREDICTION  \
73674  15_9-23         65000.0          65000               65000   
73675  15_9-23         65000.0          65000               65000   
73676  15_9-23         65000.0          65000               65000   
73677  15_9-23         65000.0          65000               70000   
73678  15_9-23         65000.0          65000               65000   

       LIGHTGBM_PREDICTION       DEPT  
73674                65000  2549.2564  
73675                65000  2549.4084  
73676                65000  2549.5604  
73677                65000  2549.7124  
73678                65000  2549.8644  

Depth range:
                 min        max  count
WELL                                  
15_9-23    2380.5748  3487.2868  11063
16_2-7           NaN        NaN      0
16_7-6           NaN        NaN      0
17_4-1           NaN        N

In [43]:
# Compare official blind well names against V2 well names

v2_blind_wells = sorted(
    v2.loc[v2["WELL"].isin(blind_wells), "WELL"].dropna().unique()
)

print("Official blind wells:")
print(sorted(blind_wells))

print("\nV2 matched blind wells:")
print(v2_blind_wells)

print("\nAll V2 wells containing the blind-well prefixes:")
for w in sorted(v2["WELL"].dropna().unique()):
    if any(str(b) in str(w) or str(w) in str(b) for b in blind_wells):
        print(repr(w))

Official blind wells:
['15_9-23', '16_2-7', '16_7-6', '17_4-1', '25_10-9', '31_2-10', '31_2-21 S', '34_3-2 S', '35_11-5', '35_9-7']

V2 matched blind wells:
['15_9-23', '16_2-7', '16_7-6', '17_4-1', '25_10-9', '31_2-10', '31_2-21 S', '34_3-2 S', '35_11-5', '35_9-7']

All V2 wells containing the blind-well prefixes:
'15_9-23'
'16_2-7'
'16_7-6'
'17_4-1'
'25_10-9'
'31_2-1'
'31_2-10'
'31_2-21 S'
'34_3-2 S'
'35_11-5'
'35_9-7'


In [44]:
# Diagnose DEPT availability in V2 for all official blind wells

dept_check = (
    v2.loc[v2["WELL"].isin(blind_wells), ["WELL", "DEPT"]]
      .groupby("WELL")["DEPT"]
      .agg(
          rows="size",
          non_null="count",
          nulls=lambda x: x.isna().sum(),
          min_depth="min",
          max_depth="max"
      )
      .reindex(sorted(blind_wells))
)

print(dept_check)

            rows  non_null  nulls    min_depth    max_depth
WELL                                                       
15_9-23    11063     11063      0  1518.280000  3212.624000
16_2-7     11683     11683      0   704.116002  2480.540002
16_7-6     10222     10222      0  1110.934400  2698.422400
17_4-1     17271     17271      0  1147.550793  3834.150793
25_10-9    10788     10788      0  1333.088800  2972.712800
31_2-10     9033      9033      0   456.224000  1829.088000
31_2-21 S   7840      7840      0  1999.928400  3191.608400
34_3-2 S   12229     12229      0  2380.574800  4321.766800
35_11-5    21770     21770      0   443.379997  3780.995997
35_9-7     10677     10677      0  1351.154800  2973.906800


In [45]:
# Safely align DEPT using WELL + within-well sample position

# Create a within-well sequential sample number
pred_depth = blind_results[["WELL", "DEPT"]].copy()

pred_depth["SAMPLE_NO"] = pred_depth.groupby("WELL").cumcount()

v2_blind_depth = (
    v2.loc[v2["WELL"].isin(blind_wells), ["WELL", "DEPT"]]
      .copy()
)

v2_blind_depth["SAMPLE_NO"] = (
    v2_blind_depth.groupby("WELL").cumcount()
)

# Merge depth using WELL + SAMPLE_NO
depth_aligned = pred_depth[["WELL", "SAMPLE_NO"]].merge(
    v2_blind_depth,
    on=["WELL", "SAMPLE_NO"],
    how="left",
    validate="one_to_one"
)

print("Aligned rows:", len(depth_aligned))
print("Missing DEPT:", depth_aligned["DEPT"].isna().sum())

print("\nDepth ranges after alignment:")
print(
    depth_aligned.groupby("WELL")["DEPT"]
    .agg(["min", "max", "count"])
)

# Verify alignment against the original prediction dataframe
blind_results["DEPT"] = depth_aligned["DEPT"].to_numpy()

print("\nFinal blind_results shape:", blind_results.shape)
print(blind_results.head())

Aligned rows: 122576
Missing DEPT: 0

Depth ranges after alignment:
                   min          max  count
WELL                                      
15_9-23    1518.280000  3212.624000  11063
16_2-7      704.116002  2480.540002  11683
16_7-6     1110.934400  2698.422400  10222
17_4-1     1147.550793  3834.150793  17271
25_10-9    1333.088800  2972.712800  10788
31_2-10     456.224000  1829.088000   9033
31_2-21 S  1999.928400  3191.608400   7840
34_3-2 S   2380.574800  4321.766800  12229
35_11-5     443.379997  3780.995997  21770
35_9-7     1351.154800  2973.906800  10677

Final blind_results shape: (122576, 6)
          WELL  TRUE_LITHOLOGY  W5_PREDICTION  XGBOOST_PREDICTION  \
73674  15_9-23         65000.0          65000               65000   
73675  15_9-23         65000.0          65000               65000   
73676  15_9-23         65000.0          65000               65000   
73677  15_9-23         65000.0          65000               70000   
73678  15_9-23         65000.0 

In [46]:
# Current FORCE 2020 penalty matrix evaluation

force_classes = [
    30000, 65030, 65000, 80000, 74000, 70000,
    70032, 88000, 86000, 99000, 90000, 93000
]

penalty_matrix = np.array([
    [0,   2,   3,   3.5, 2.5, 3.5, 3,   3,   3,   3,   3,   3],
    [2,   0,   2,   3.5, 2.5, 3.5, 3,   3,   3,   3,   3,   3],
    [3,   2,   0,   3.5, 2.5, 3.5, 3,   3,   3,   3,   3,   3],
    [3.5, 3.5, 3.5, 0,   3,   2,   2.5, 3,   3,   3,   3,   3],
    [2.5, 2.5, 2.5, 3,   0,   3,   2.5, 3,   3,   3,   3,   3],
    [3.5, 3.5, 3.5, 2,   3,   0,   2,   3,   3,   3,   3,   3],
    [3,   3,   3,   2.5, 2.5, 2,   0,   3,   3,   3,   3,   3],
    [3,   3,   3,   3,   3,   3,   3,   0,   2,   3,   3,   3],
    [3,   3,   3,   3,   3,   3,   3,   3,   0,   3,   3,   3],
    [3,   3,   3,   3,   3,   3,   3,   3,   3,   0,   3,   3],
    [3,   3,   3,   3,   3,   3,   3,   3,   3,   3,   0,   3],
    [3,   3,   3,   3,   3,   3,   3,   3,   3,   3,   3,   0]
])

def force_score(y_true, y_pred):
    class_to_idx = {c: i for i, c in enumerate(force_classes)}

    true_idx = np.array([class_to_idx.get(int(x), -1) for x in y_true])
    pred_idx = np.array([class_to_idx.get(int(x), -1) for x in y_pred])

    valid = (true_idx >= 0) & (pred_idx >= 0)

    penalties = penalty_matrix[true_idx[valid], pred_idx[valid]]

    return -penalties.mean()

force_scores = {
    "W5": force_score(
        blind_results["TRUE_LITHOLOGY"],
        blind_results["W5_PREDICTION"]
    ),
    "XGBoost": force_score(
        blind_results["TRUE_LITHOLOGY"],
        blind_results["XGBOOST_PREDICTION"]
    ),
    "LightGBM": force_score(
        blind_results["TRUE_LITHOLOGY"],
        blind_results["LIGHTGBM_PREDICTION"]
    )
}

print("Current FORCE scores:")
for model, score in force_scores.items():
    print(f"{model:10s}: {score:.6f}")

Current FORCE scores:
W5        : -0.691942
XGBoost   : -0.840923
LightGBM  : -0.789278
